# 10 · Обучение XLS-R
Начните с Frozen Encoder. Длительное обучение запускается только вручную.

In [ ]:
import subprocess
import sys
from pathlib import Path

repo=Path('/content/audio-deepfake-detector')
if not (repo/'.git').exists(): subprocess.run(['git','clone','https://github.com/Unvand/audio-deepfake-detector.git',str(repo)],check=True)
else: subprocess.run(['git','-C',str(repo),'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(repo/'requirements-colab-train.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.44,<5'],check=True)
sys.path.insert(0,str(repo))
commit=subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
from google.colab import drive

drive.mount('/content/drive')
base=Path('/content/drive/MyDrive/VKR_RuASD')
prepared=base/'prepared_v1'
print('Git commit:',commit)

In [ ]:
import torch

from audio_deepfake_detector.src.xlsr_training import XLSRConfig, audit_manifest, train

frame,summary=audit_manifest(prepared/'ruasd_segments_manifest.csv')
display(summary)
print('GPU:',torch.cuda.is_available())
print('XLS-R 300m может не поместиться на T4 с большим batch; начните с batch=1–2 и accumulation.')

In [ ]:
import ipywidgets as widgets

mode=widgets.Dropdown(options=['frozen','partial'],value='frozen',description='Режим')
batch=widgets.IntSlider(value=2,min=1,max=8,description='Batch')
epochs=widgets.IntSlider(value=5,min=1,max=50,description='Эпохи')
augment=widgets.FloatSlider(value=0,min=0,max=.5,step=.05,description='Аугментации')
resume=widgets.Checkbox(value=False,description='Resume')
run_id=widgets.Text(value='xlsr_baseline_v1',description='run_id')
display(mode,batch,epochs,augment,resume,run_id)

In [ ]:
# GPU/CPU smoke-test: одна малая запись, без сохранения эксперимента
RUN_SMOKE_TEST = False
if RUN_SMOKE_TEST:
    import gc
    import hashlib

    from audio_deepfake_detector.src.xlsr_training import WaveformDataset, XLSRDetector

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    cfg = XLSRConfig(batch_size=1, epochs=1, mode="frozen")
    small = frame.loc[frame.split.eq("train")].groupby("label", group_keys=False).head(1)
    model = None
    try:
        model = XLSRDetector.from_pretrained(cfg).to(device)
        model.encoder.eval()
        wave, mask, label, _ = WaveformDataset(small, prepared, "train", cfg)[0]
        wave, mask, label = wave[None].to(device), mask[None].to(device), label[None].to(device)
        before_head = model.head[1].weight.detach().clone()
        def encoder_digest():
            digest = hashlib.sha256()
            for value in model.encoder.state_dict().values():
                digest.update(value.detach().cpu().contiguous().numpy().tobytes())
            return digest.hexdigest()
        before_encoder = encoder_digest()
        optimizer = torch.optim.AdamW(model.head.parameters(), lr=cfg.head_lr)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(model(wave, mask), label)
        loss.backward()
        assert all(p.grad is None for p in model.encoder.parameters())
        optimizer.step()
        assert not torch.equal(before_head, model.head[1].weight)
        assert before_encoder == encoder_digest()
        print('Устройство:', device)
        print('GPU:', torch.cuda.get_device_name(device) if device.type == 'cuda' else 'нет')
        print('Smoke-test завершён; loss:', loss.detach().item())
        print('Пиковая память GPU (МиБ):', round(torch.cuda.max_memory_allocated(device) / 2**20, 1) if device.type == 'cuda' else 'N/A')
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError('Недостаточно памяти CUDA. Освободите GPU и повторите с одной записью.') from exc
    finally:
        del model
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()

In [ ]:
# Пробное обучение XLS-R на небольшой выборке RuASD
RUN_PILOT_TRAINING = False
if RUN_PILOT_TRAINING:
    from audio_deepfake_detector.src.xlsr_training import select_pilot_sources

    pilot = select_pilot_sources(frame, train_per_class=500, validation_per_class=100, seed=42)
    print('Фактический состав pilot:')
    display(pilot.groupby(['split', 'label']).agg(segments=('segment_id', 'size'), sources=('sample_id', 'nunique')))
    cfg = XLSRConfig(mode='frozen', batch_size=2, accumulation=4, epochs=3,
                     head_lr=0.001, augmentation_probability=0., seed=42, threshold=.5)
    output_dir = base / 'training_xlsr_v1/runs/xlsr_pilot_v1'
    output_dir.mkdir(parents=True, exist_ok=True) if not output_dir.exists() else None
    # Новый каталог обязателен: train() не перезаписывает существующий запуск.
    if any(output_dir.iterdir()):
        raise FileExistsError(f'Пробный запуск уже существует: {output_dir}. Выберите Resume отдельно.')
    pilot[['sample_id', 'segment_id', 'split', 'label', 'audio_path']].to_csv(output_dir / 'pilot_selection.csv', index=False)
    pilot_history = train(prepared, pilot, output_dir, cfg, resume=False, git_commit=commit)
    print('Пробный запуск завершён:', output_dir)

In [ ]:
RUN_TRAINING=False
if RUN_TRAINING:
 cfg=XLSRConfig(mode=mode.value,batch_size=batch.value,epochs=epochs.value,augmentation_probability=augment.value)
 out=base/'training_xlsr_v1/runs'/run_id.value.strip()
 history=train(prepared,frame,out,cfg,resume=resume.value,git_commit=commit)
 print('Сохранено:',out,'эпох:',len(history))